# 01 · Ingestão Bronze

Leitura pura da Landing + gravação em Delta na Bronze, **sem nenhuma regra de negócio**.

Decisões desta camada (ver `data-engineering/AGENTS.md`):
- **Tudo é lido como texto** (sem `inferSchema`). Os dados da V-Energy têm datas em formatos mistos, decimais com vírgula e códigos que poderiam perder zeros à esquerda, a inferência alteraria o conteúdo original, e o case exige preservá-lo exatamente.
- Colunas de auditoria em todas as tabelas:
  - `source_file`: arquivo de origem (essencial para as leituras, que vêm em 5 arquivos)
  - `ingestion_timestamp`: momento da gravação. Como na aula, é adicionado **na mesma cadeia que termina no `.write()`** — por causa da *lazy evaluation*, ele só é calculado quando a escrita dispara.
- Escrita com `mode("overwrite")`: o notebook pode ser reexecutado do zero sem duplicar dados.

| Fonte | Formato observado na inspeção (notebook 00) | Tabela Bronze |
|---|---|---|
| `unidades_consumidoras.csv` | CSV, separador `,` | `unidades_consumidoras` |
| `leituras/leituras_20XX.csv` (5 arquivos) | CSV, separador `,`, mesmo cabeçalho | `leituras` |
| `faturas.csv` | CSV, separador **`;`** | `faturas` |
| `medidores.csv` | CSV, separador `,` | `medidores` |
| `ocorrencias_rede.csv` | CSV, separador `,` | `ocorrencias_rede` |
| `geracao_distribuida.csv` | CSV, separador `,` | `geracao_distribuida` |
| `atendimentos.json` | JSON **array multilinha**, com lista aninhada `historico` | `atendimentos` |
| `inspecoes.xlsx` | Excel com 3 abas (`ago-set`, `out-nov`, `dez`), mesmo layout, cabeçalho na 1ª linha | `inspecoes` (abas unidas + coluna `aba_origem`) |

In [ ]:
%pip install openpyxl -q

## 1. Parâmetros do ambiente

In [ ]:
catalog = "v_energy"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

## 2. Funções auxiliares

Todas as leituras usam as mesmas opções, então centralizamos em funções para garantir que as 8 fontes sejam tratadas de forma idêntica.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from functools import reduce
import pandas as pd

def ler_csv(caminho: str, sep: str = ","):
    """Lê CSV(s) como texto puro, preservando o conteúdo original, e registra o arquivo de origem."""
    return (
        spark.read
        .option("header", True)
        .option("sep", sep)
        .option("encoding", "UTF-8")
        .option("quote", '"')
        .option("escape", '"')
        .csv(caminho)                                   # sem inferSchema: todas as colunas são string
        .select("*", F.col("_metadata.file_path").alias("source_file"))
    )

def gravar_bronze(df, tabela: str):
    """Adiciona o timestamp de ingestão na mesma cadeia do write e grava em Delta (overwrite)."""
    (
        df.withColumn("ingestion_timestamp", F.current_timestamp())
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{bronze_schema}.{tabela}")
    )
    total = spark.table(f"{bronze_schema}.{tabela}").count()
    print(f"[OK] {bronze_schema}.{tabela}: {total:,} linhas".replace(",", "."))

## 3. Arquivos CSV

Seis fontes em CSV. Atenção: `faturas.csv` usa **ponto e vírgula** como separador (o sistema de faturamento exporta assim, como a Diretora de Tecnologia comentou no case).

In [ ]:
gravar_bronze(ler_csv(f"{landing_path}/unidades_consumidoras.csv"), "unidades_consumidoras")
gravar_bronze(ler_csv(f"{landing_path}/leituras/*.csv"),             "leituras")
gravar_bronze(ler_csv(f"{landing_path}/faturas.csv", sep=";"),       "faturas")
gravar_bronze(ler_csv(f"{landing_path}/medidores.csv"),              "medidores")
gravar_bronze(ler_csv(f"{landing_path}/ocorrencias_rede.csv"),       "ocorrencias_rede")
gravar_bronze(ler_csv(f"{landing_path}/geracao_distribuida.csv"),    "geracao_distribuida")

## 4. JSON de atendimentos

O arquivo é um **array JSON formatado em várias linhas**, por isso `multiLine=True`.
- `primitivesAsString=True` mantém números (ex.: `nota_satisfacao`) como texto, seguindo a regra da Bronze.
- A lista aninhada `historico` (tratativas de cada chamado) é **preservada como está** (array de structs). Ela será aberta em tabela própria na Silver.

In [ ]:
df_atendimentos = (
    spark.read
    .option("multiLine", True)
    .option("primitivesAsString", True)
    .json(f"{landing_path}/atendimentos.json")
    .select("*", F.col("_metadata.file_path").alias("source_file"))
)

gravar_bronze(df_atendimentos, "atendimentos")
df_atendimentos.printSchema()

## 5. Excel de inspeções

O Spark não lê `.xlsx` nativamente, então usamos **pandas** apenas para a leitura (arquivo pequeno, ~1,4 mil linhas) e convertemos para Spark.
- As 3 abas têm o **mesmo layout** (períodos diferentes), então são unidas em uma única tabela, com a coluna `aba_origem` para não perder a informação de qual aba veio cada linha.
- `dtype=str` mantém tudo como texto; células vazias viram `null`.

In [ ]:
caminho_inspecoes = f"{landing_path}/inspecoes.xlsx"
abas = pd.read_excel(caminho_inspecoes, sheet_name=None, dtype=str)

frames = []
for nome_aba, pdf in abas.items():
    pdf.columns = [str(c).strip() for c in pdf.columns]
    pdf = pdf.astype(object).where(pdf.notna(), None)              # NaN do pandas -> null
    schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
    sdf = (
        spark.createDataFrame(pdf, schema=schema)
        .withColumn("aba_origem", F.lit(nome_aba))
        .withColumn("source_file", F.lit(caminho_inspecoes))
    )
    print(f"Aba '{nome_aba}': {len(pdf)} linhas, colunas = {list(pdf.columns)}")
    frames.append(sdf)

df_inspecoes = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), frames)
gravar_bronze(df_inspecoes, "inspecoes")

## 6. Validação da ingestão

Duas checagens:
1. **Integridade da leitura (CSV):** o número de linhas na tabela deve ser igual ao número de linhas de dados do arquivo (total de linhas − cabeçalhos − linhas em branco). Se diferir, alguma linha foi quebrada ou mal interpretada na leitura.
2. **Volume esperado:** comparação com os volumes informados no documento do case.

O resultado é gravado em `bronze.controle_ingestao` para auditoria.

In [ ]:
fontes_csv = {
    "unidades_consumidoras": f"{landing_path}/unidades_consumidoras.csv",
    "leituras":              f"{landing_path}/leituras/*.csv",
    "faturas":               f"{landing_path}/faturas.csv",
    "medidores":             f"{landing_path}/medidores.csv",
    "ocorrencias_rede":      f"{landing_path}/ocorrencias_rede.csv",
    "geracao_distribuida":   f"{landing_path}/geracao_distribuida.csv",
}

# Volumes aproximados informados no documento do case
volume_esperado = {
    "unidades_consumidoras": 46_000,
    "leituras":              2_180_000,
    "faturas":               2_180_000,
    "medidores":             52_000,
    "ocorrencias_rede":      180_000,
    "geracao_distribuida":   3_500,
    "atendimentos":          319_000,
    "inspecoes":             1_400,
}

resultados = []
for tabela, esperado in volume_esperado.items():
    linhas_tabela = spark.table(f"{bronze_schema}.{tabela}").count()

    linhas_arquivo = None
    if tabela in fontes_csv:
        txt = spark.read.text(fontes_csv[tabela]).filter(F.trim("value") != "")
        n_arquivos = txt.select("_metadata.file_path").distinct().count()
        linhas_arquivo = txt.count() - n_arquivos                     # desconta 1 cabeçalho por arquivo

    leitura_ok = (linhas_arquivo is None) or (linhas_arquivo == linhas_tabela)
    desvio_pct = round(100 * (linhas_tabela - esperado) / esperado, 1)

    resultados.append((tabela, linhas_tabela, linhas_arquivo, leitura_ok, esperado, desvio_pct))
    status = "OK" if leitura_ok else "VERIFICAR"
    print(f"[{status}] {tabela:<22} tabela={linhas_tabela:>10,}  arquivo={str(linhas_arquivo):>10}  esperado≈{esperado:>10,}  desvio={desvio_pct:+}%")

df_controle = (
    spark.createDataFrame(
        resultados,
        "tabela string, linhas_tabela long, linhas_arquivo long, leitura_ok boolean, volume_esperado long, desvio_pct double",
    )
    .withColumn("verificado_em", F.current_timestamp())
)
df_controle.write.format("delta").mode("append").saveAsTable(f"{bronze_schema}.controle_ingestao")
display(df_controle)

## 7. Amostra das tabelas Bronze

In [ ]:
for tabela in volume_esperado:
    print(f"=== {bronze_schema}.{tabela} ===")
    display(spark.table(f"{bronze_schema}.{tabela}").limit(5))

## Resumo

| Camada | O que este notebook entrega |
|---|---|
| Landing | 12 arquivos brutos no Volume `/Volumes/v_energy/bronze/landing` |
| Bronze | 8 tabelas Delta em `v_energy.bronze`, conteúdo original preservado como texto, com `source_file` e `ingestion_timestamp` |
| Controle | `v_energy.bronze.controle_ingestao` com contagens e checagem de integridade de cada execução |

